In [7]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support
)

# Load datasets
reason_train = pd.read_csv("part1_train.csv")
reason_val = pd.read_csv("part1_val.csv")
reason_test = pd.read_csv("part1_test.csv")

labels = [
    "Safety", "Funding", "Efficacy",
    "Recruitment", "Other"
]

emb_cols = [f"emb_{i}" for i in range(150)]

# Identify labeled examples
labeled = (reason_train[labels] != -100).all(axis=1)

# Extract embeddings
X_train_raw = reason_train[emb_cols].to_numpy()
X_val_raw = reason_val[emb_cols].to_numpy()
X_test_raw = reason_test[emb_cols].to_numpy()

# Standardize using training data only
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

# Ground-truth labels
y_train = reason_train.loc[labeled, labels].astype(int)
y_val = reason_val[labels].astype(int)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Labeled samples:", labeled.sum())
print("Unlabeled samples:", (~labeled).sum())

Train: (1171, 150)
Validation: (78, 150)
Test: (312, 150)
Labeled samples: 157
Unlabeled samples: 1014


In [8]:
# Cluster all training examples, including unlabeled trials
kmeans = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=10
)

cluster_train = kmeans.fit_predict(X_train)
cluster_val = kmeans.predict(X_val)

Y_known = y_train.to_numpy(dtype=int)
known_clusters = cluster_train[labeled.to_numpy()]

# Estimate five reason probabilities for each cluster
global_rate = Y_known.mean(axis=0)
cluster_rates = np.zeros((10, len(labels)))

for k in range(10):
    members = Y_known[known_clusters == k]

    # Smooth estimates for clusters with few labeled trials
    cluster_rates[k] = (
        (members.sum(axis=0) + 5 * global_rate)
        / (len(members) + 5)
    )

# Predict missing labels for unlabeled training examples
unlabeled = ~labeled.to_numpy()
pseudo_labels = (
    cluster_rates[cluster_train[unlabeled]] >= 0.5
).astype(int)

# Combine true labels and pseudo-labels
Y_augmented = np.zeros(
    (len(reason_train), len(labels)),
    dtype=int
)

Y_augmented[labeled.to_numpy()] = Y_known
Y_augmented[unlabeled] = pseudo_labels

# Train a classifier on the augmented dataset
augmented_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=500,
        solver="liblinear",
        random_state=42
    )
)

augmented_model.fit(X_train, Y_augmented)

augmented_val_pred = augmented_model.predict(X_val)

print("K-Means Augmented Model")
print("Sample-wise F1:",
      round(f1_score(y_val, augmented_val_pred,
                     average="samples", zero_division=0), 4))
print("Macro F1:",
      round(f1_score(y_val, augmented_val_pred,
                     average="macro", zero_division=0), 4))

print("\nPer-label F1:")
print(pd.Series(
    f1_score(y_val, augmented_val_pred,
             average=None, zero_division=0),
    index=labels
).round(4))


C:\Users\keche\anaconda3\envs\cs5785\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(


K-Means Augmented Model
Sample-wise F1: 0.7308
Macro F1: 0.391

Per-label F1:
Safety         0.2000
Funding        0.0000
Efficacy       0.6400
Recruitment    0.9333
Other          0.1818
dtype: float64


In [9]:

from sklearn.metrics import precision_recall_fscore_support

print("Validation label distribution:")
print(y_val.sum())

print("\nPredicted positive counts:")
print(pd.Series(
    augmented_val_pred.sum(axis=0),
    index=labels
))

precision, recall, f1, support = precision_recall_fscore_support(
    y_val,
    augmented_val_pred,
    average=None,
    zero_division=0
)

results = pd.DataFrame({
    "Reason": labels,
    "Support": support,
    "Predicted Positive": augmented_val_pred.sum(axis=0),
    "Precision": precision,
    "Recall": recall,
    "F1": f1
})

print("\nPer-label evaluation:")
display(results.round(4))

print("\nAverage number of labels per trial:")
print("True:", y_val.to_numpy().sum(axis=1).mean())
print("Predicted:", augmented_val_pred.sum(axis=1).mean())


Validation label distribution:
Safety          9
Funding         4
Efficacy       14
Recruitment    50
Other          10
dtype: int64

Predicted positive counts:
Safety          1
Funding         0
Efficacy       11
Recruitment    55
Other           1
dtype: int64

Per-label evaluation:


,Reason,Support,Predicted Positive,Precision,Recall,F1
0,Safety,9,1,1.0000,0.1111,0.2000
1,Funding,4,0,0.0000,0.0000,0.0000
2,Efficacy,14,11,0.7273,0.5714,0.6400
3,Recruitment,50,55,0.8909,0.9800,0.9333
4,Other,10,1,1.0000,0.1000,0.1818



Average number of labels per trial:
True: 1.1153846153846154
Predicted: 0.8717948717948718


In [10]:
# Predicted probabilities
val_probs = augmented_model.predict_proba(X_val)

# Compare a few shared thresholds
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

rows = []

for threshold in thresholds:
    pred = (val_probs >= threshold).astype(int)

    rows.append({
        "Threshold": threshold,
        "Sample-wise F1": f1_score(
            y_val, pred,
            average="samples",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_val, pred,
            average="macro",
            zero_division=0
        ),
        "Avg Labels": pred.sum(axis=1).mean()
    })

threshold_results = pd.DataFrame(rows)

display(threshold_results.round(4))

best_row = threshold_results.loc[
    threshold_results["Sample-wise F1"].idxmax()
]

best_threshold = float(best_row["Threshold"])

print("Best shared threshold:", best_threshold)

tuned_pred = (val_probs >= best_threshold).astype(int)

print("\nPer-label F1 after tuning:")
print(pd.Series(
    f1_score(
        y_val,
        tuned_pred,
        average=None,
        zero_division=0
    ),
    index=labels
).round(4))


,Threshold,Sample-wise F1,Macro F1,Avg Labels
0,0.20,0.7778,0.5430,1.0513
1,0.25,0.7735,0.5103,1.0385
2,0.30,0.7543,0.4980,1.0000
3,0.35,0.7372,0.4668,0.9487
4,0.40,0.7308,0.4613,0.9231
5,0.45,0.7308,0.3910,0.8718
6,0.50,0.7308,0.3910,0.8718


Best shared threshold: 0.2

Per-label F1 after tuning:
Safety         0.3636
Funding        0.4000
Efficacy       0.7500
Recruitment    0.9159
Other          0.2857
dtype: float64


In [11]:

from sklearn.mixture import GaussianMixture
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

Y_known = y_train.to_numpy(dtype=int)
known_mask = labeled.to_numpy()
Y_val = y_val.to_numpy(dtype=int)

results = []

for n_components in [5, 10, 15]:
    # PCA helps stabilize GMM covariance estimation
    from sklearn.decomposition import PCA

    pca = PCA(n_components=20, random_state=42)
    X_gmm = pca.fit_transform(X_train)

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="diag",
        reg_covar=1e-4,
        random_state=42,
        n_init=3
    )

    responsibilities = gmm.fit_predict(X_gmm)
    soft_memberships = gmm.predict_proba(X_gmm)

    # Estimate label probabilities within each component
    R_known = soft_memberships[known_mask]
    global_rate = Y_known.mean(axis=0)

    component_rates = (
        R_known.T @ Y_known + 5 * global_rate
    ) / (
        R_known.sum(axis=0)[:, None] + 5
    )

    pseudo_prob = (
        soft_memberships[~known_mask] @ component_rates
    )

    Y_aug = np.zeros(
        (len(reason_train), len(labels)),
        dtype=int
    )

    Y_aug[known_mask] = Y_known
    Y_aug[~known_mask] = (
        pseudo_prob >= 0.5
    ).astype(int)

    model = OneVsRestClassifier(
        LogisticRegression(
            max_iter=500,
            solver="liblinear",
            random_state=42
        )
    )

    model.fit(X_train, Y_aug)

    val_probs_gmm = model.predict_proba(X_val)

    for threshold in [0.2, 0.3, 0.4, 0.5]:
        preds = (
            val_probs_gmm >= threshold
        ).astype(int)

        results.append({
            "Components": n_components,
            "Threshold": threshold,
            "Sample F1": f1_score(
                Y_val, preds,
                average="samples",
                zero_division=0
            ),
            "Macro F1": f1_score(
                Y_val, preds,
                average="macro",
                zero_division=0
            )
        })

gmm_results = pd.DataFrame(results)

display(
    gmm_results.sort_values(
        "Sample F1",
        ascending=False
    ).round(4)
)


C:\Users\keche\anaconda3\envs\cs5785\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(
C:\Users\keche\anaconda3\envs\cs5785\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(
C:\Users\keche\anaconda3\envs\cs5785\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(
C:\Users\keche\anaconda3\envs\cs5785\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning:

,Components,Threshold,Sample F1,Macro F1
8,15,0.2,0.7179,0.4746
4,10,0.2,0.7115,0.4909
9,15,0.3,0.7009,0.4276
5,10,0.3,0.6923,0.4443
10,15,0.4,0.6752,0.3761
6,10,0.4,0.6709,0.3974
11,15,0.5,0.6624,0.2991
7,10,0.5,0.6581,0.3204
0,5,0.2,0.6197,0.4611
1,5,0.3,0.5812,0.3865


In [12]:

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

baseline_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=42
    )
)

baseline_model.fit(
    X_train[labeled.to_numpy()],
    y_train.to_numpy()
)

baseline_probs = baseline_model.predict_proba(X_val)

baseline_results = []

for threshold in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    pred = (baseline_probs >= threshold).astype(int)

    baseline_results.append({
        "Threshold": threshold,
        "Sample F1": f1_score(
            y_val, pred,
            average="samples",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_val, pred,
            average="macro",
            zero_division=0
        )
    })

baseline_results = pd.DataFrame(baseline_results)

display(
    baseline_results.sort_values(
        "Sample F1",
        ascending=False
    ).round(4)
)


,Threshold,Sample F1,Macro F1
6,0.50,0.8654,0.7067
2,0.30,0.8504,0.6720
5,0.45,0.8500,0.6707
1,0.25,0.8457,0.6625
4,0.40,0.8432,0.6500
3,0.35,0.8419,0.6395
0,0.20,0.8218,0.6268


In [13]:
# Supervised baseline probabilities
p_supervised = baseline_model.predict_proba(X_val)

# K-Means augmented probabilities
p_kmeans = augmented_model.predict_proba(X_val)

results = []

for weight in [0.0, 0.1, 0.2, 0.3, 0.5]:
    # weight = contribution from K-Means augmented model
    p_ensemble = (
        (1 - weight) * p_supervised
        + weight * p_kmeans
    )

    for threshold in [0.3, 0.4, 0.5]:
        pred = (p_ensemble >= threshold).astype(int)

        results.append({
            "KMeans Weight": weight,
            "Threshold": threshold,
            "Sample F1": f1_score(
                y_val, pred,
                average="samples",
                zero_division=0
            ),
            "Macro F1": f1_score(
                y_val, pred,
                average="macro",
                zero_division=0
            )
        })

ensemble_results = pd.DataFrame(results)

display(
    ensemble_results.sort_values(
        "Sample F1",
        ascending=False
    ).round(4)
)


,KMeans Weight,Threshold,Sample F1,Macro F1
13,0.5,0.4,0.8718,0.7182
12,0.5,0.3,0.8671,0.7055
2,0.0,0.5,0.8654,0.7067
7,0.2,0.4,0.8654,0.6992
3,0.1,0.3,0.8632,0.6876
10,0.3,0.4,0.8611,0.6900
11,0.3,0.5,0.8611,0.7248
8,0.2,0.5,0.8568,0.7100
9,0.3,0.3,0.8538,0.6611
5,0.1,0.5,0.8526,0.6949


In [14]:

# Save current best model configuration
best_weight = 0.5
best_threshold = 0.4

best_val_probs = (
    (1 - best_weight) * baseline_model.predict_proba(X_val)
    + best_weight * augmented_model.predict_proba(X_val)
)

best_val_pred = (
    best_val_probs >= best_threshold
).astype(int)

print("Final candidate validation metrics:")
print("Sample-wise F1:",
      f1_score(y_val, best_val_pred,
               average="samples", zero_division=0))
print("Macro F1:",
      f1_score(y_val, best_val_pred,
               average="macro", zero_division=0))

print("\nPer-label F1:")
print(pd.Series(
    f1_score(y_val, best_val_pred,
             average=None, zero_division=0),
    index=labels
).round(4))


Final candidate validation metrics:
Sample-wise F1: 0.8717948717948718
Macro F1: 0.718181160991831

Per-label F1:
Safety         0.8889
Funding        0.3333
Efficacy       0.7586
Recruitment    0.9434
Other          0.6667
dtype: float64


In [15]:

from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np

# Standardize year using training data only
year_scaler = StandardScaler()

year_train = year_scaler.fit_transform(
    reason_train[["start_year"]]
)

year_val = year_scaler.transform(
    reason_val[["start_year"]]
)

# Combine embeddings and start year
X_train_year = np.hstack([X_train, year_train])
X_val_year = np.hstack([X_val, year_val])

# Train on true labeled examples only
year_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=42
    )
)

year_model.fit(
    X_train_year[labeled.to_numpy()],
    y_train.to_numpy()
)

year_probs = year_model.predict_proba(X_val_year)

for threshold in [0.3, 0.4, 0.5]:
    pred = (year_probs >= threshold).astype(int)

    print(
        f"Threshold={threshold:.1f} | "
        f"Sample F1={f1_score(y_val, pred, average='samples', zero_division=0):.4f} | "
        f"Macro F1={f1_score(y_val, pred, average='macro', zero_division=0):.4f}"
    )


Threshold=0.3 | Sample F1=0.6530 | Macro F1=0.4813
Threshold=0.4 | Sample F1=0.7248 | Macro F1=0.5431
Threshold=0.5 | Sample F1=0.7500 | Macro F1=0.5712


In [16]:
# Current best ensemble
test_probs = (
    0.5 * baseline_model.predict_proba(X_test)
    + 0.5 * augmented_model.predict_proba(X_test)
)

test_pred = (test_probs >= 0.4).astype(int)

# Convert multi-label predictions to Kaggle format
def format_reasons(row):
    return " ".join(
        label for label, value in zip(labels, row)
        if value == 1
    )

submission = pd.DataFrame({
    "ID": reason_test["ID"],
    "Predicted": [
        format_reasons(row) for row in test_pred
    ]
})

submission.to_csv("part1_submission.csv", index=False)

print("Submission shape:", submission.shape)
print("\nPredicted positive counts:")
print(pd.Series(test_pred.sum(axis=0), index=labels))

print("\nPreview:")
display(submission.head())

print("\nEmpty predictions:",
      (submission["Predicted"] == "").sum())


Submission shape: (312, 2)

Predicted positive counts:
Safety          62
Funding         16
Efficacy        60
Recruitment    196
Other           70
dtype: int64

Preview:


,ID,Predicted
0,TR00756,Safety Efficacy
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Efficacy Recruitment



Empty predictions: 4


In [17]:
check = pd.read_csv("part1_submission.csv", keep_default_na=False)

assert len(check) == len(reason_test) == 312
assert list(check.columns) == ["ID", "Predicted"]
assert check["ID"].equals(reason_test["ID"])
assert check["ID"].is_unique

allowed = set(labels)
assert all(
    set(s.split()).issubset(allowed)
    for s in check["Predicted"]
)

print("Submission format OK!")

Submission format OK!


In [18]:

import numpy as np
import pandas as pd

# Recreate the best ensemble predictions
test_probs = (
    0.5 * baseline_model.predict_proba(X_test)
    + 0.5 * augmented_model.predict_proba(X_test)
)

test_pred = (test_probs >= 0.4).astype(int)

# Find rows with no predicted labels
empty_mask = test_pred.sum(axis=1) == 0

print("Empty predictions before:", empty_mask.sum())

# Assign the most probable reason to each empty row
empty_indices = np.where(empty_mask)[0]

for i in empty_indices:
    best_label = np.argmax(test_probs[i])
    test_pred[i, best_label] = 1

# Convert to submission format
def format_reasons(row):
    return " ".join(
        label for label, value in zip(labels, row)
        if value == 1
    )

submission = pd.DataFrame({
    "ID": reason_test["ID"],
    "Predicted": [
        format_reasons(row) for row in test_pred
    ]
})

# Save corrected submission
submission.to_csv(
    "part1_submission_fixed.csv",
    index=False
)

print("Empty predictions after:",
      (submission["Predicted"] == "").sum())

print("Missing values:",
      submission.isna().sum().to_dict())

print("Shape:", submission.shape)
display(submission.head())


Empty predictions before: 4
Empty predictions after: 0
Missing values: {'ID': 0, 'Predicted': 0}
Shape: (312, 2)


,ID,Predicted
0,TR00756,Safety Efficacy
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Efficacy Recruitment
